In [ ]:
import os
import time
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# Frozen experiment configuration
MODELS = {
    "LightGBM": "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",
    "RF": "Prescriptive/RF_prescriptive_optimizer_inputs.csv",
    "XGBoost": "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv",
}

COHORT_PATH = "Cohort/fixed_cohort.csv"
PRODUCTION_SUMMARY = "Results/all_models_summary.csv"
OUTPUT_DIR = "Results"

N_CASES = 12
N_ROOMS = 3
ROOM_CAPACITY = 480
TURNOVER = 20
BALANCE_WEIGHT = 0.10
DURATION_CAP = 360.0
TIME_LIMIT = 300

# Representative cases: low risk + primary comparison.
# Add 1.0 only if you want a high-risk diagnostic after the initial audit.
AUDIT_LAMBDAS = [0.0, 0.5]

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Audit configuration loaded.")
print("Models:", list(MODELS))
print("Lambdas:", AUDIT_LAMBDAS)
print("Planned diagnostic runs:", len(MODELS) * len(AUDIT_LAMBDAS))


In [ ]:
# ============================================================
# Load and protect the frozen cohort + verified model inputs
# ============================================================
assert os.path.exists(COHORT_PATH), f"Missing frozen cohort: {COHORT_PATH}"
cohort = pd.read_csv(COHORT_PATH)
assert "LOG_ID" in cohort.columns
assert len(cohort) == N_CASES
assert cohort["LOG_ID"].is_unique
frozen_ids = cohort["LOG_ID"].astype(str).tolist()

verified = {}
for model_name, path in MODELS.items():
    assert os.path.exists(path), f"Missing {model_name} input: {path}"
    df = pd.read_csv(path)
    required = {"LOG_ID", "DURATION_P50_MINS", "DURATION_P90_MINS"}
    assert required.issubset(df.columns), f"{model_name}: missing columns {required-set(df.columns)}"
    assert df["LOG_ID"].is_unique
    df = df.copy()
    df["LOG_ID"] = df["LOG_ID"].astype(str)
    missing = set(frozen_ids) - set(df["LOG_ID"])
    assert not missing, f"{model_name}: missing frozen IDs {missing}"
    day = df.set_index("LOG_ID").loc[frozen_ids].reset_index()
    assert day["LOG_ID"].tolist() == frozen_ids
    verified[model_name] = day
    print(f"{model_name}: PASS | full={df.shape} | fixed={day.shape}")

print("\nFrozen-cohort integrity: PASS")


In [ ]:
def solve_or_milp_semantics_audit(
    df_day,
    lam,
    n_rooms=3,
    room_capacity=480,
    turnover=20,
    balance_weight=0.10,
    duration_cap=360.0,
    time_limit=300,
):
    N_CASES_LOCAL = len(df_day)
    if N_CASES_LOCAL == 0:
        raise ValueError("df_day must contain at least one surgery.")

    I = range(N_CASES_LOCAL)
    R = range(n_rooms)
    P50 = df_day["DURATION_P50_MINS"].to_numpy(dtype=float)
    P90 = df_day["DURATION_P90_MINS"].to_numpy(dtype=float)

    duration_uncapped = P50 + lam * (P90 - P50)
    if duration_cap is None:
        duration = duration_uncapped.copy()
        n_capped = 0
    else:
        cap_mask = duration_uncapped > duration_cap
        n_capped = int(cap_mask.sum())
        duration = np.minimum(duration_uncapped, duration_cap)

    SERIAL_HORIZON = float(np.sum(duration) + turnover * max(N_CASES_LOCAL - 1, 0))
    DAY_END = 1440
    max_duration = float(np.max(duration))
    M = float(DAY_END + max_duration + turnover)

    m = gp.Model(f"semantics_audit_{lam:.2f}")
    m.Params.OutputFlag = 0
    m.Params.MIPGap = 0.01
    m.Params.TimeLimit = time_limit
    m.Params.Heuristics = 0.4

    x = m.addVars(I, R, vtype=GRB.BINARY, name="x")
    s = m.addVars(I, lb=0, ub=DAY_END, vtype=GRB.CONTINUOUS, name="start")
    c = m.addVars(I, lb=0, vtype=GRB.CONTINUOUS, name="completion")
    C = m.addVars(R, lb=0, ub=SERIAL_HORIZON, vtype=GRB.CONTINUOUS, name="room_finish")
    OT = m.addVars(R, lb=0, vtype=GRB.CONTINUOUS, name="overtime")
    IDLE = m.addVars(R, lb=0, vtype=GRB.CONTINUOUS, name="idle")
    system_makespan = m.addVar(lb=0, ub=SERIAL_HORIZON, vtype=GRB.CONTINUOUS, name="makespan")
    min_room_load = m.addVar(lb=0, ub=SERIAL_HORIZON, vtype=GRB.CONTINUOUS, name="min_load")

    y = {}
    for i in I:
        for j in I:
            if i < j:
                for r in R:
                    y[i, j, r] = m.addVar(vtype=GRB.BINARY, name=f"y_{i}_{j}_{r}")

    # Assignment
    for i in I:
        m.addConstr(gp.quicksum(x[i, r] for r in R) == 1, name=f"assign_{i}")

    # Endogenous pairwise sequencing (same structure as production Big-M engine)
    for i in I:
        for j in I:
            if i >= j:
                continue
            for r in R:
                m.addConstr(
                    s[i] + duration[i] + turnover <= s[j]
                    + M * (1 - y[i, j, r])
                    + M * (2 - x[i, r] - x[j, r]),
                    name=f"seq_ij_{i}_{j}_{r}",
                )
                m.addConstr(
                    s[j] + duration[j] + turnover <= s[i]
                    + M * y[i, j, r]
                    + M * (2 - x[i, r] - x[j, r]),
                    name=f"seq_ji_{i}_{j}_{r}",
                )
                m.addConstr(y[i, j, r] <= x[i, r], name=f"link_i_{i}_{j}_{r}")
                m.addConstr(y[i, j, r] <= x[j, r], name=f"link_j_{i}_{j}_{r}")

    for i in I:
        m.addConstr(c[i] == s[i] + duration[i], name=f"completion_{i}")

    # Current room-finish semantics: lower bound only
    for r in R:
        for i in I:
            m.addConstr(C[r] >= c[i] - M * (1 - x[i, r]), name=f"room_finish_{i}_{r}")

    # Symmetry breaking
    for r in range(n_rooms - 1):
        m.addConstr(C[r] >= C[r + 1], name=f"room_symmetry_{r}")

    for r in R:
        m.addConstr(system_makespan >= C[r], name=f"makespan_room_{r}")
        m.addConstr(min_room_load <= C[r], name=f"min_load_room_{r}")

    total_duration = float(np.sum(duration))
    m.addConstr(system_makespan >= total_duration / n_rooms, name="makespan_workload_lb")

    for r in R:
        m.addConstr(OT[r] - IDLE[r] == C[r] - room_capacity, name=f"shift_balance_{r}")

    objective = (
        gp.quicksum(OT[r] for r in R)
        + 0.25 * gp.quicksum(IDLE[r] for r in R)
        + 0.5 * system_makespan
        + balance_weight * (system_makespan - min_room_load)
    )
    m.setObjective(objective, GRB.MINIMIZE)

    t0 = time.time()
    m.optimize()
    wall = time.time() - t0
    if m.SolCount == 0:
        return None

    assignment = {i: next(r for r in R if x[i, r].X > 0.5) for i in I}
    starts = {i: float(s[i].X) for i in I}
    completions = {i: float(c[i].X) for i in I}

    return {
        "Lambda": float(lam),
        "Objective": float(m.ObjVal),
        "Gap": float(m.MIPGap),
        "Status": int(m.Status),
        "Solve_Time": float(wall),
        "N_Capped": n_capped,
        "Planning_Duration": {i: float(duration[i]) for i in I},
        "Room_Assignment": assignment,
        "Start_Times": starts,
        "Completion_Times": completions,
        "Room_Finish_C": {r: float(C[r].X) for r in R},
        "Room_OT": {r: float(OT[r].X) for r in R},
        "Room_IDLE": {r: float(IDLE[r].X) for r in R},
        "System_Makespan": float(system_makespan.X),
        "Min_Room_Load_Var": float(min_room_load.X),
        "Variables": int(m.NumVars),
        "Constraints": int(m.NumConstrs),
        "Nodes": float(m.NodeCount),
    }


In [ ]:
def build_semantics_tables(model_name, df_day, result, room_capacity=480, turnover=20, tol=1e-5):
    """Reconstruct actual room chronology and compare alternative metric semantics."""
    rows = []
    case_rows = []
    n_rooms = len(result["Room_Finish_C"])
    assignment = result["Room_Assignment"]
    starts = result["Start_Times"]
    completions = result["Completion_Times"]
    durations = result["Planning_Duration"]

    for r in range(n_rooms):
        cases = [i for i, rr in assignment.items() if rr == r]
        cases = sorted(cases, key=lambda i: starts[i])

        for i in cases:
            case_rows.append({
                "Model": model_name,
                "Lambda": result["Lambda"],
                "Room": r,
                "Case_Index": i,
                "LOG_ID": df_day.iloc[i]["LOG_ID"],
                "Start": starts[i],
                "Duration": durations[i],
                "Completion": completions[i],
            })

        actual_last_completion = max((completions[i] for i in cases), default=0.0)
        first_start = min((starts[i] for i in cases), default=0.0)
        c_var = result["Room_Finish_C"][r]
        c_inflation = c_var - actual_last_completion

        # Occupied workload: surgery durations + turnover only between assigned cases.
        occupied_workload = sum(durations[i] for i in cases) + turnover * max(len(cases) - 1, 0)

        # Internal/pre-start gaps observed in the actual clock-time schedule.
        if cases:
            chronological_span = actual_last_completion - first_start
            internal_gap = max(chronological_span - occupied_workload, 0.0)
        else:
            chronological_span = 0.0
            internal_gap = 0.0

        current_ot = result["Room_OT"][r]
        current_idle = result["Room_IDLE"][r]

        actual_finish_ot = max(actual_last_completion - room_capacity, 0.0)
        actual_finish_idle = max(room_capacity - actual_last_completion, 0.0)

        workload_ot = max(occupied_workload - room_capacity, 0.0)
        workload_unused_capacity = max(room_capacity - occupied_workload, 0.0)

        rows.append({
            "Model": model_name,
            "Lambda": result["Lambda"],
            "Room": r,
            "N_Cases": len(cases),
            "Case_Indices": ",".join(map(str, cases)),
            "First_Start": first_start,
            "Actual_Last_Completion": actual_last_completion,
            "MILP_C": c_var,
            "C_Minus_ActualFinish": c_inflation,
            "C_Inflated": bool(c_inflation > tol),
            "Occupied_Workload": occupied_workload,
            "Chronological_Span": chronological_span,
            "Internal_or_PreStart_Gap": internal_gap,
            "Current_OT_from_C": current_ot,
            "Current_IDLE_from_C": current_idle,
            "ActualFinish_OT": actual_finish_ot,
            "ActualFinish_IDLE": actual_finish_idle,
            "Workload_OT": workload_ot,
            "Workload_Unused_Capacity": workload_unused_capacity,
            "CurrentIdle_Minus_WorkloadUnused": current_idle - workload_unused_capacity,
        })

    return pd.DataFrame(rows), pd.DataFrame(case_rows)


In [ ]:
# ============================================================
# Run representative semantics audit
# ============================================================
room_tables = []
case_tables = []
run_records = []

production = None
if os.path.exists(PRODUCTION_SUMMARY):
    production = pd.read_csv(PRODUCTION_SUMMARY)
    print(f"Loaded production summary: {production.shape}")
else:
    print("Production summary not found; objective cross-check will be skipped.")

for model_name in MODELS:
    df_day = verified[model_name]
    for lam in AUDIT_LAMBDAS:
        print("\n" + "=" * 70)
        print(f"AUDIT | {model_name} | lambda={lam:.1f}")
        print("=" * 70)

        result = solve_or_milp_semantics_audit(
            df_day=df_day,
            lam=float(lam),
            n_rooms=N_ROOMS,
            room_capacity=ROOM_CAPACITY,
            turnover=TURNOVER,
            balance_weight=BALANCE_WEIGHT,
            duration_cap=DURATION_CAP,
            time_limit=TIME_LIMIT,
        )
        assert result is not None, f"No feasible incumbent for {model_name}, lambda={lam}"

        room_df, case_df = build_semantics_tables(model_name, df_day, result, ROOM_CAPACITY, TURNOVER)
        room_tables.append(room_df)
        case_tables.append(case_df)

        prod_obj = np.nan
        if production is not None and {"Model", "Lambda", "Objective"}.issubset(production.columns):
            match = production[(production["Model"] == model_name) & (np.isclose(production["Lambda"].astype(float), lam))]
            if len(match) == 1:
                prod_obj = float(match.iloc[0]["Objective"])

        run_records.append({
            "Model": model_name,
            "Lambda": lam,
            "Audit_Objective": result["Objective"],
            "Production_Objective": prod_obj,
            "Objective_Difference": result["Objective"] - prod_obj if np.isfinite(prod_obj) else np.nan,
            "Gap": result["Gap"],
            "Status": result["Status"],
            "Solve_Time": result["Solve_Time"],
            "Any_C_Inflation": bool(room_df["C_Inflated"].any()),
            "Max_C_Inflation": float(room_df["C_Minus_ActualFinish"].max()),
            "Total_Current_IDLE": float(room_df["Current_IDLE_from_C"].sum()),
            "Total_Workload_Unused_Capacity": float(room_df["Workload_Unused_Capacity"].sum()),
            "Total_Current_OT": float(room_df["Current_OT_from_C"].sum()),
            "Total_Workload_OT": float(room_df["Workload_OT"].sum()),
        })

        print(room_df[[
            "Room", "N_Cases", "Actual_Last_Completion", "MILP_C",
            "C_Minus_ActualFinish", "Occupied_Workload",
            "Current_OT_from_C", "Current_IDLE_from_C",
            "Workload_OT", "Workload_Unused_Capacity"
        ]].to_string(index=False))

room_audit_df = pd.concat(room_tables, ignore_index=True)
case_audit_df = pd.concat(case_tables, ignore_index=True)
run_audit_df = pd.DataFrame(run_records)


In [ ]:
# ============================================================
# Final diagnostic summary + exports
# ============================================================
print("\n" + "=" * 70)
print("ROOM METRIC SEMANTICS AUDIT — SUMMARY")
print("=" * 70)

summary_cols = [
    "Model", "Lambda", "Audit_Objective", "Production_Objective",
    "Objective_Difference", "Any_C_Inflation", "Max_C_Inflation",
    "Total_Current_IDLE", "Total_Workload_Unused_Capacity",
    "Total_Current_OT", "Total_Workload_OT", "Gap", "Status"
]
print(run_audit_df[summary_cols].to_string(index=False))

n_inflated = int(room_audit_df["C_Inflated"].sum())
max_inflation = float(room_audit_df["C_Minus_ActualFinish"].max())

print("\nKey checks")
print(f"Rooms audited                  : {len(room_audit_df)}")
print(f"Rooms with C inflation         : {n_inflated}")
print(f"Maximum C - actual finish      : {max_inflation:.6f} min")
print(f"Any C inflation                : {bool(n_inflated > 0)}")

# A small tolerance is appropriate for solver numerics.
if n_inflated == 0:
    print("\nC-FINISH CHECK: PASS for audited incumbents.")
    print("No audited room has MILP C materially above reconstructed last completion.")
else:
    print("\nC-FINISH CHECK: ATTENTION REQUIRED.")
    print("At least one audited room has MILP C above reconstructed last completion.")

# Workload semantic difference is reported, not automatically labelled an error:
# it depends on what the dissertation intends 'idle' and 'workload' to mean.
room_audit_df["Abs_CurrentIdle_vs_WorkloadUnused"] = (
    room_audit_df["Current_IDLE_from_C"] - room_audit_df["Workload_Unused_Capacity"]
).abs()

print("\nLargest |current idle - workload unused capacity|:",
      f"{room_audit_df['Abs_CurrentIdle_vs_WorkloadUnused'].max():.6f} min")

room_path = os.path.join(OUTPUT_DIR, "room_metric_semantics_audit_rooms.csv")
case_path = os.path.join(OUTPUT_DIR, "room_metric_semantics_audit_cases.csv")
run_path = os.path.join(OUTPUT_DIR, "room_metric_semantics_audit_runs.csv")

room_audit_df.to_csv(room_path, index=False)
case_audit_df.to_csv(case_path, index=False)
run_audit_df.to_csv(run_path, index=False)

print("\nSaved:")
print(" -", room_path)
print(" -", case_path)
print(" -", run_path)
